In [1]:
# Mulligan rules:
# 1.- Initial Draw 7 cards. If number of card types criteria is not met, return them to the deck and shuffle.
# 2.- First free Draw 7 cards. If number of card types criteria is not met, return them to the deck and shuffle.
# 3.- Take one last mulligan and Draw 7 cards and accept always. Return one to the bottom of the deck with the
#     following priority: Others > Bombs > Ramp > Land


# Gameplan sequence

# Turn 0: 
# Draw 7 (Cards in hand = 7; Total cards seen = 7)
# Do mulligan once. If not fulfilling requirements, do it again. Regardless of the second result, go ahead.

# Turn 1
# Draw 1 (Cards in hand = 8; Total cards seen = 8)
# Play a land (Cards in hand = 7; Total cards seen = 8)
# If available, play a 1CMC draw spell. If so, add 1 to the number of seen cards onwards.

# Turn 2
# Draw 1 (Cards in hand = 8; Total cards seen = 9)
# Play a land (Cards in hand = 7; Total cards seen = 9)
# Play a ramp piece (Cards in hand = 6; Total cards seen = 9)

#Turn 3:
# Draw 1 (Cards in hand = 7; Total cards seen = 10)
# Play A&N (no effect in hand)

#Turn 4:
# Draw 1 (Cards in hand = 8; Total cards seen = 11)
# Play Bomb (Cards in hand = 7; Total cards seen = 11)

In [2]:
import numpy as np
import time
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # registers the 3D projection
from numba import njit, prange
from scipy.interpolate import griddata
from lib.sim_lib import simulate_games, print_simulation_report
from lib.analysis_lib import LRBD_grid_search, analyze_sensitivity

N = 99 #always 99 cards in deck

In [3]:
# --- Numba compilation of simulation function --- #
L = 0      # lands
R = 0      # ramp
B = 0      # bombs
D = 0      # draw 
t_land = 0  # minimum lands to keep hand
t_ramp = 0  # minimum ramp to keep hand
t_bomb = 0  # minimum bombs to keep hand
t_draw = 0  # minimum draw to keep hand
extra_mulligan = True  # enable or disable extra mulligan logic

# --- Build the deck template ---
base_deck = np.concatenate([
    np.ones(L, dtype=np.uint8),             # land card type = 1
    np.full(R, 2, dtype=np.uint8),          # ramp card type = 2
    np.full(B, 3, dtype=np.uint8),          # bomb card type = 3
    np.full(D, 4, dtype=np.uint8),          # draw card type = 4
    np.zeros(N - L - R - B - D, dtype=np.uint8) # filler / other cards = 0
])

# --- Game plan (per turn list of card types to play) ---
gameplan = [
    np.array([1, 4], dtype=np.uint8),   # turn 1: play land + optional draw
    np.array([1, 2], dtype=np.uint8),   # turn 2: play land and ramp
    np.array([1], dtype=np.uint8),      # turn 3: play land
    np.array([1, 3], dtype=np.uint8)    # turn 4: play land and bomb
]

# --- Run simulation once (for Numba compilation) ---
result, fails, mulligans = simulate_games(1, base_deck, t_land, t_ramp, t_bomb, t_draw, gameplan, extra_mulligan)
print("Compilation OK")

Compilation OK


In [4]:
# --- Run full simulation with user input parameters ---
L = 38 # number of lands in deck
R = 15 # number of ramp pieces in deck
B = 17 # number of bombs in deck
D = 5
t_land = 2  # minimum lands to keep hand
t_ramp = 1  # minimum ramp to keep hand
t_bomb = 0  # minimum bombs to keep hand
t_draw = 0
N_sim = 100_000  # number of simulations
extra_mulligan = True  # enable or disable extra mulligan logic

# --- Build the deck template ---
base_deck = np.concatenate([
    np.ones(L, dtype=np.uint8),             # land card type = 1
    np.full(R, 2, dtype=np.uint8),          # ramp card type = 2
    np.full(B, 3, dtype=np.uint8),          # bomb card type = 3
    np.full(D, 4, dtype=np.uint8),          # draw card type = 4
    np.zeros(N - L - R - B - D, dtype=np.uint8) # filler / other cards = 0
])

result, fail_summary, mulligans = simulate_games(N_sim, base_deck, t_land, t_ramp, t_bomb, t_draw, gameplan, extra_mulligan)
print_simulation_report(result, fail_summary, mulligans, N_sim)


✅ Success rate: 62.26%
❌ Failure rate: 37.74%

🎲 Mulligan usage breakdown:
0 mulligan(s): 55363 games → 55.36%
1 mulligan(s): 24946 games → 24.95%
2 mulligan(s): 19691 games → 19.69%

🔎 Failure breakdown by operation (as % of ALL games):
 Turn 1 - Play Land:  0.391%
 Turn 2 - Play Land:  0.955%
 Turn 2 - Play Ramp:  3.877%
 Turn 3 - Play Land:  7.562%
 Turn 4 - Play Land: 13.807%
 Turn 4 - Play Bomb: 11.148%

📊 Conditional breakdown (as % of FAILED games):
 Turn 1 - Play Land:   1.04%
 Turn 2 - Play Land:   2.53%
 Turn 2 - Play Ramp:  10.27%
 Turn 3 - Play Land:  20.04%
 Turn 4 - Play Land:  36.58%
 Turn 4 - Play Bomb:  29.54%


In [5]:
# Optimization search
Lmin = 33
Lmax = 40
Rmin = 12
Rmax = 18
Bmin = 12
Bmax = 18
Dmin = 0
Dmax = 10

t_land = 2  # minimum lands to keep hand
t_ramp = 1  # minimum ramp to keep hand
t_bomb = 0  # minimum bombs to keep hand
t_draw = 0
N_sim = 100_000  # number of simulations
extra_mulligan = True  #enable or disable extra mulligan logic

data, best_config, best_rate, best_fail_summary = LRBD_grid_search(Lmin, Lmax, Rmin, Rmax, Bmin, Bmax, Dmin, Dmax,
                                                                       N, t_land, t_ramp, t_bomb, t_draw, N_sim,
                                                                       gameplan, extra_mulligan=True)

dic = analyze_sensitivity(data)


🚀 Starting optimization over 4312 combinations...

Checked 100/4312 configs (Best: L=33, R=12, B=18, D=10 → 51.01%) Elapsed: 3.1s
Checked 200/4312 configs (Best: L=33, R=13, B=18, D=10 → 52.27%) Elapsed: 5.9s
Checked 300/4312 configs (Best: L=33, R=14, B=18, D=10 → 53.57%) Elapsed: 8.7s
Checked 400/4312 configs (Best: L=33, R=16, B=18, D=10 → 55.30%) Elapsed: 11.4s
Checked 500/4312 configs (Best: L=33, R=17, B=18, D=10 → 56.29%) Elapsed: 14.0s
Checked 600/4312 configs (Best: L=33, R=18, B=18, D=10 → 56.86%) Elapsed: 16.8s
Checked 700/4312 configs (Best: L=33, R=18, B=18, D=10 → 56.86%) Elapsed: 19.6s
Checked 800/4312 configs (Best: L=33, R=18, B=18, D=10 → 56.86%) Elapsed: 22.5s
Checked 900/4312 configs (Best: L=34, R=15, B=18, D=10 → 56.95%) Elapsed: 25.3s
Checked 1000/4312 configs (Best: L=34, R=17, B=18, D=9 → 58.14%) Elapsed: 28.0s
Checked 1100/4312 configs (Best: L=34, R=18, B=18, D=10 → 59.15%) Elapsed: 30.6s
Checked 1200/4312 configs (Best: L=34, R=18, B=18, D=10 → 59.15%) Elap